# NB4 — Visualisations
**Zombie Firms Replication** — *Geographical Analysis of the Italian Industrial North*

Publication-quality figures for the full national panel (2016–2024, 80,682 firms).
All charts use a consistent navy/red palette matching the thesis LaTeX style.

**Reads:** `zombie_panel_classified.parquet` (output of NB1/NB2)

**Writes:** All figures to `output/figures/`

In [ ]:
from pathlib import Path

BASE_DIR = Path("/Users/leoss/Desktop/Thesis Replication/output")
FIG_DIR  = BASE_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

PRIMARY_ZOMBIE = "zombie_mcgowan"
MAP_YEAR       = 2022

# Palette — matches thesis navy/red
NAVY   = "#1B2A4A"
RED    = "#C0392B"
TEAL   = "#1A6B72"
GOLD   = "#B8860B"
LGREY  = "#F4F4F4"
MGREY  = "#CCCCCC"

print(f"Output: {FIG_DIR}")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

# Use Helvetica-like font stack; fall back gracefully
mpl.rcParams.update({
    "font.family"       : "sans-serif",
    "font.sans-serif"   : ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : False,
    "figure.dpi"        : 150,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

df = pd.read_parquet(BASE_DIR / "zombie_panel_classified.parquet")
print(f"Loaded: {len(df):,} rows | {df['bvd_id'].nunique():,} firms")

# Convenience: analysis years only
dfa = df[df["year"].between(2016, 2024)].copy()


## Fig 1 — Zombie share over time, three definitions

Tracks all three classification approaches annually. The McGowan series is the primary definition; the Storz and weak ICR series provide comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ts = (
    dfa[dfa[["zombie_mcgowan","zombie_weak","zombie_storz"]].notna().any(axis=1)]
    .groupby("year")
    .agg(
        mcgowan = ("zombie_mcgowan", "mean"),
        weak    = ("zombie_weak",    "mean"),
        storz   = ("zombie_storz",   "mean"),
    )
) * 100

ax.fill_between(ts.index, ts["weak"],    alpha=0.08, color=NAVY)
ax.fill_between(ts.index, ts["storz"],   alpha=0.08, color=TEAL)
ax.fill_between(ts.index, ts["mcgowan"], alpha=0.12, color=RED)

ax.plot(ts.index, ts["weak"],    color=NAVY, lw=1.4, ls="--", label="Weak (ICR < 1)")
ax.plot(ts.index, ts["storz"],   color=TEAL, lw=1.4, ls=":",  label="Storz (ICR < 1, consecutive)")
ax.plot(ts.index, ts["mcgowan"], color=RED,  lw=2.2,           label="McGowan (primary)")

# COVID annotation
ax.axvspan(2020, 2021, alpha=0.06, color="grey", zorder=0)
ax.text(2020.4, ts["weak"].max()*0.92, "COVID-19", fontsize=7.5,
        color="grey", va="top")

ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.set_xlabel("Year", fontsize=10, labelpad=6)
ax.set_ylabel("Share of firms (%)", fontsize=10, labelpad=6)
ax.set_xticks(ts.index)
ax.tick_params(axis="x", rotation=0, labelsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left")

# Subtle horizontal baseline
ax.axhline(0, color=MGREY, lw=0.6)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_zombie_share_time.png")
plt.show()
print("Saved: fig1_zombie_share_time.png")


## Fig 2 — Zombie stock, entry and exit (McGowan)

Shows the gross flows into and out of zombie status each year alongside the end-of-year stock. Highlights persistence vs. turnover.

In [ ]:
# Build entry/exit table from NB1-style logic
zdf = dfa[["bvd_id","year","zombie_mcgowan"]].dropna().sort_values(["bvd_id","year"])
zdf["z_lag"] = zdf.groupby("bvd_id")["zombie_mcgowan"].shift(1)
zdf["entry"] = ((zdf["zombie_mcgowan"]==1) & (zdf["z_lag"]==0)).astype(int)
zdf["exit"]  = ((zdf["zombie_mcgowan"]==0) & (zdf["z_lag"]==1)).astype(int)

flows = (
    zdf.groupby("year")
    .agg(stock=("zombie_mcgowan","sum"),
         entry=("entry","sum"),
         exit=("exit","sum"),
         n=("bvd_id","nunique"))
    .assign(share=lambda d: d["stock"]/d["n"]*100)
)
flows = flows[flows.index >= 2018]  # need lag

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

w = 0.25
xs = np.arange(len(flows))

ax1.bar(xs - w, flows["entry"], width=w*1.8, color=RED,  alpha=0.75, label="Entries")
ax1.bar(xs + w, flows["exit"],  width=w*1.8, color=TEAL, alpha=0.75, label="Exits")
ax2.plot(xs, flows["share"], color=NAVY, lw=2.2, marker="o",
         markersize=5, label="Zombie share (right)")

ax1.set_xticks(xs)
ax1.set_xticklabels(flows.index.astype(int), fontsize=9)
ax1.set_ylabel("Number of firms", fontsize=10, labelpad=6)
ax2.set_ylabel("Zombie share (%)", fontsize=10, labelpad=6, color=NAVY)
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax2.tick_params(axis="y", colors=NAVY)
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color(NAVY)

handles = [
    mpatches.Patch(color=RED,  alpha=0.75, label="New zombie entries"),
    mpatches.Patch(color=TEAL, alpha=0.75, label="Zombie exits"),
    Line2D([0],[0], color=NAVY, lw=2, marker="o", markersize=5, label="Zombie share"),
]
ax1.legend(handles=handles, frameon=False, fontsize=9, loc="upper left")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_zombie_entry_exit.png")
plt.show()
print("Saved: fig2_zombie_entry_exit.png")


## Fig 3 — Zombie share by 2-digit NACE sector (2022)

Horizontal bar chart ordered by zombie share. Bubble size proportional to firm count.

In [ ]:
SECTOR_LABELS = {
    10:"Food products", 11:"Beverages", 12:"Tobacco", 13:"Textiles",
    14:"Wearing apparel", 15:"Leather", 16:"Wood products", 17:"Paper",
    18:"Printing", 19:"Coke/petroleum", 20:"Chemicals", 21:"Pharma",
    22:"Rubber/plastics", 23:"Non-metallic minerals", 24:"Basic metals",
    25:"Fabricated metals", 26:"Computer/electronics", 27:"Electrical equipment",
    28:"Machinery & equipment", 29:"Motor vehicles", 30:"Other transport",
    31:"Furniture", 32:"Other manufacturing", 33:"Repair/installation",
}

sec = (
    dfa[(dfa["year"]==MAP_YEAR) & dfa["zombie_mcgowan"].notna()]
    .groupby("nace_2digit")
    .agg(n_firms=("bvd_id","nunique"),
         zombie_share=("zombie_mcgowan","mean"))
    .reset_index()
)
sec["label"] = sec["nace_2digit"].map(SECTOR_LABELS).fillna(sec["nace_2digit"].astype(str))
sec = sec[sec["n_firms"] >= 50].sort_values("zombie_share")

fig, ax = plt.subplots(figsize=(7, 7))

colors = [RED if x >= sec["zombie_share"].quantile(0.75) else
          NAVY if x <= sec["zombie_share"].quantile(0.25) else
          "#5B7FA6" for x in sec["zombie_share"]]

bars = ax.barh(sec["label"], sec["zombie_share"]*100,
               color=colors, edgecolor="white", linewidth=0.4, height=0.7)

# Value labels
for bar, val in zip(bars, sec["zombie_share"]*100):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", ha="left", fontsize=7.5, color="#444")

ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.set_xlabel("Zombie share — McGowan (%)", fontsize=10, labelpad=6)
ax.tick_params(axis="y", labelsize=8.5)
ax.axvline(sec["zombie_share"].mean()*100, color=MGREY, lw=1, ls="--", zorder=0)
ax.text(sec["zombie_share"].mean()*100 + 0.05,
        len(sec)-0.5, "Mean", fontsize=7.5, color="grey")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_zombie_by_sector.png")
plt.show()
print("Saved: fig3_zombie_by_sector.png")


## Fig 4 — Province × year heatmap

Shows zombie share for the top provinces over time as a heatmap. Useful for spotting spatial persistence.

In [ ]:
prov_yr = (
    dfa[dfa["zombie_mcgowan"].notna()]
    .groupby(["province","year"])["zombie_mcgowan"]
    .mean() * 100
).unstack("year")

# Keep provinces with enough firms
n_prov = dfa[dfa["year"]==MAP_YEAR].groupby("province")["bvd_id"].nunique()
top_provs = n_prov[n_prov >= 200].index
prov_yr = prov_yr.loc[prov_yr.index.isin(top_provs)]

# Sort by 2022 zombie share
prov_yr = prov_yr.sort_values(MAP_YEAR, ascending=False)

fig, ax = plt.subplots(figsize=(9, max(5, len(prov_yr)*0.32)))

im = ax.imshow(prov_yr.values, aspect="auto",
               cmap="RdYlBu_r", vmin=0, vmax=prov_yr.values.max())

ax.set_xticks(range(len(prov_yr.columns)))
ax.set_xticklabels(prov_yr.columns.astype(int), fontsize=9)
ax.set_yticks(range(len(prov_yr)))
ax.set_yticklabels(prov_yr.index, fontsize=8.5)

# Cell annotations
for i in range(len(prov_yr)):
    for j in range(len(prov_yr.columns)):
        val = prov_yr.values[i, j]
        if not np.isnan(val):
            c = "white" if val > prov_yr.values.max()*0.65 else "#333"
            ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                    fontsize=6.5, color=c)

cb = plt.colorbar(im, ax=ax, shrink=0.5, pad=0.01)
cb.ax.tick_params(labelsize=8)
cb.set_label("Zombie share (%)", fontsize=9)

ax.spines[["top","right","bottom","left"]].set_visible(False)
ax.tick_params(length=0)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_province_year_heatmap.png")
plt.show()
print("Saved: fig4_province_year_heatmap.png")


## Fig 5 — Zombie firm lifecycle: financial trajectory

Tracks median ICR, ROA, and investment rate for firms in the years before, during, and after zombie classification. Uses an event-study window: t=0 is first year classified as zombie.

In [ ]:
# Event study: t=0 is first year a firm is classified zombie
zombie_firms = dfa[dfa["zombie_mcgowan"]==1]["bvd_id"].unique()

first_zombie = (
    dfa[dfa["zombie_mcgowan"]==1]
    .groupby("bvd_id")["year"].min()
    .rename("event_year")
    .reset_index()
)

evdf = dfa.merge(first_zombie, on="bvd_id", how="inner")
evdf["t"] = evdf["year"] - evdf["event_year"]
evdf = evdf[evdf["t"].between(-3, 4)]

metrics = {"icr": "ICR", "roa": "ROA", "investment_rate": "Investment rate"}

fig, axes = plt.subplots(1, 3, figsize=(11, 4))

for ax, (col, label) in zip(axes, metrics.items()):
    traj = evdf.groupby("t")[col].agg(["median","sem"]).reset_index()
    traj = traj[traj["t"].between(-3,4)]

    ax.fill_between(traj["t"],
                    traj["median"] - 1.96*traj["sem"],
                    traj["median"] + 1.96*traj["sem"],
                    alpha=0.12, color=NAVY)
    ax.plot(traj["t"], traj["median"], color=NAVY, lw=2)

    # Shade zombie period (t=0 onward)
    ymin, ymax = ax.get_ylim()
    ax.axvspan(0, 4, alpha=0.06, color=RED, zorder=0)
    ax.axvline(0, color=RED, lw=1, ls="--", alpha=0.7)
    ax.text(0.15, 0.96, "Zombie period", transform=ax.transAxes,
            fontsize=7.5, color=RED, va="top")

    ax.set_xlabel("Years relative to first zombie classification", fontsize=9)
    ax.set_ylabel(f"Median {label}", fontsize=9)
    ax.set_xticks(range(-3, 5))
    ax.axhline(0, color=MGREY, lw=0.6)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_zombie_lifecycle.png")
plt.show()
print("Saved: fig5_zombie_lifecycle.png")


## Fig 6 — Size distribution: zombie vs. non-zombie firms

Kernel density of log total assets for zombie and non-zombie firms in the cross-section year, showing whether zombies cluster at a particular size range.

In [ ]:
from scipy.stats import gaussian_kde

snap = dfa[(dfa["year"]==MAP_YEAR) & dfa["zombie_mcgowan"].notna() &
           dfa["total_assets"].notna() & (dfa["total_assets"]>0)].copy()
snap["log_ta"] = np.log(snap["total_assets"])

z   = snap[snap["zombie_mcgowan"]==1]["log_ta"].values
nz  = snap[snap["zombie_mcgowan"]==0]["log_ta"].values

xs = np.linspace(snap["log_ta"].quantile(0.01),
                 snap["log_ta"].quantile(0.99), 300)

kde_z  = gaussian_kde(z,  bw_method=0.25)
kde_nz = gaussian_kde(nz, bw_method=0.25)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(xs, kde_nz(xs), alpha=0.15, color=NAVY)
ax.fill_between(xs, kde_z(xs),  alpha=0.20, color=RED)
ax.plot(xs, kde_nz(xs), color=NAVY, lw=2,   label=f"Non-zombie (n={len(nz):,})")
ax.plot(xs, kde_z(xs),  color=RED,  lw=2,   label=f"Zombie (n={len(z):,})")

# Median lines
ax.axvline(np.median(nz), color=NAVY, lw=1, ls="--", alpha=0.6)
ax.axvline(np.median(z),  color=RED,  lw=1, ls="--", alpha=0.6)

# x-axis: convert log back to readable EUR labels
xticks = [np.log(v) for v in [100, 500, 2000, 10000, 50000, 200000]]
xlabels = ["100", "500", "2k", "10k", "50k", "200k"]
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_xlabel("Total assets (€ thousands)", fontsize=10, labelpad=6)
ax.set_ylabel("Density", fontsize=10, labelpad=6)
ax.legend(frameon=False, fontsize=9)
ax.set_xlim(snap["log_ta"].quantile(0.01), snap["log_ta"].quantile(0.99))

plt.tight_layout()
plt.savefig(FIG_DIR / "fig6_size_distribution.png")
plt.show()
print("Saved: fig6_size_distribution.png")


## Fig 7 — North vs. South zombie share over time

Splits the panel by NUTS1 macro-region to show whether zombification trends differ between the industrial North and the rest of Italy.

In [ ]:
# NUTS2 → macro region
def macro_region(code):
    if pd.isna(code): return "Unknown"
    if code.startswith("ITC") or code.startswith("ITH"): return "North"
    if code.startswith("ITI"): return "Centre"
    return "South & Islands"

dfa["macro"] = dfa["nuts2_code"].apply(macro_region)

ts_macro = (
    dfa[dfa["zombie_mcgowan"].notna()]
    .groupby(["year","macro"])["zombie_mcgowan"]
    .mean() * 100
).unstack("macro")

fig, ax = plt.subplots(figsize=(8, 4))

palette = {"North": NAVY, "Centre": TEAL, "South & Islands": RED}
for col, color in palette.items():
    if col not in ts_macro.columns: continue
    ax.plot(ts_macro.index, ts_macro[col], color=color,
            lw=2.2 if col=="North" else 1.6,
            marker="o", markersize=4, label=col)
    ax.fill_between(ts_macro.index, ts_macro[col], alpha=0.06, color=color)

ax.axvspan(2020, 2021, alpha=0.06, color="grey", zorder=0)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax.set_xlabel("Year", fontsize=10, labelpad=6)
ax.set_ylabel("Zombie share — McGowan (%)", fontsize=10, labelpad=6)
ax.set_xticks(ts_macro.index)
ax.legend(frameon=False, fontsize=9)
ax.axhline(0, color=MGREY, lw=0.6)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig7_north_south.png")
plt.show()
print("Saved: fig7_north_south.png")


## Fig 8 — Firm age: zombie vs. non-zombie

Stacked bar chart showing the age distribution (decile bins) of zombie and non-zombie firms. Tests whether zombies are disproportionately old or young.

In [ ]:
snap = dfa[(dfa["year"]==MAP_YEAR) & dfa["zombie_mcgowan"].notna() &
           dfa["firm_age"].notna()].copy()

bins   = [0, 5, 10, 15, 20, 30, 40, 50, 200]
labels = ["0-5","6-10","11-15","16-20","21-30","31-40","41-50","50+"]
snap["age_bin"] = pd.cut(snap["firm_age"], bins=bins, labels=labels, right=True)

age_dist = (
    snap.groupby(["age_bin","zombie_mcgowan"], observed=True)["bvd_id"]
    .nunique().unstack("zombie_mcgowan")
    .fillna(0)
    .rename(columns={0.0: "Non-zombie", 1.0: "Zombie"})
)
age_pct = age_dist.div(age_dist.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(age_pct))
w = 0.4

ax.bar(x - w/2, age_pct["Non-zombie"], width=w, color=NAVY,  alpha=0.85, label="Non-zombie")
ax.bar(x + w/2, age_pct["Zombie"],     width=w, color=RED,   alpha=0.85, label="Zombie")

# Overlay zombie share line
ax2 = ax.twinx()
zshare = snap.groupby("age_bin", observed=True)["zombie_mcgowan"].mean() * 100
ax2.plot(x, zshare.values, color=GOLD, lw=1.8, marker="D",
         markersize=5, label="Zombie share (right)")
ax2.set_ylabel("Zombie share within age group (%)", fontsize=9, color=GOLD)
ax2.tick_params(axis="y", colors=GOLD)
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color(GOLD)

ax.set_xticks(x)
ax.set_xticklabels(age_pct.index, fontsize=9)
ax.set_xlabel("Firm age (years)", fontsize=10, labelpad=6)
ax.set_ylabel("Share of firms in age group (%)", fontsize=10, labelpad=6)

handles = [
    mpatches.Patch(color=NAVY, alpha=0.85, label="Non-zombie"),
    mpatches.Patch(color=RED,  alpha=0.85, label="Zombie"),
    Line2D([0],[0], color=GOLD, lw=1.8, marker="D", markersize=5, label="Zombie share"),
]
ax.legend(handles=handles, frameon=False, fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig8_age_distribution.png")
plt.show()
print("Saved: fig8_age_distribution.png")


## Fig 9 — Province-level zombie congestion vs. investment rate

Scatter of province-year observations. Bubble size = number of firms. Fits a local regression line to show unconditional relationship.

In [ ]:
from scipy.stats import linregress

prov = (
    dfa[dfa["zombie_mcgowan"].notna() & dfa["investment_rate"].notna()]
    .groupby(["province","year"])
    .agg(
        zombie_share  = ("zombie_mcgowan",   "mean"),
        inv_rate_med  = ("investment_rate",  "median"),
        n_firms       = ("bvd_id",           "nunique"),
    )
    .reset_index()
)
prov = prov[prov["n_firms"] >= 50]

x = prov["zombie_share"].values * 100
y = prov["inv_rate_med"].values * 100

# Winsorise for display
x = np.clip(x, np.percentile(x,1), np.percentile(x,99))
y = np.clip(y, np.percentile(y,1), np.percentile(y,99))

slope, intercept, r, p, _ = linregress(x, y)

fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(x, y, s=prov["n_firms"]/8, alpha=0.35,
                c=prov["year"], cmap="Blues", edgecolors="none")

# OLS line
xfit = np.linspace(x.min(), x.max(), 100)
ax.plot(xfit, intercept + slope*xfit, color=RED, lw=1.8, ls="--",
        label=f"OLS: β={slope:.3f}, R²={r**2:.3f}, p={p:.3f}")

cb = plt.colorbar(sc, ax=ax, shrink=0.5, pad=0.01)
cb.set_label("Year", fontsize=9)
cb.ax.tick_params(labelsize=8)

ax.set_xlabel("Zombie share — province-year (%)", fontsize=10, labelpad=6)
ax.set_ylabel("Median investment rate (%)", fontsize=10, labelpad=6)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax.legend(frameon=False, fontsize=9)
ax.axhline(0, color=MGREY, lw=0.6)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig9_congestion_investment.png")
plt.show()
print("Saved: fig9_congestion_investment.png")


## Summary

In [ ]:
figs = sorted(FIG_DIR.glob("fig*.png"))
print(f"Figures saved to {FIG_DIR}:")
for f in figs:
    print(f"  {f.name}")
